In [3]:
import pandas as pd
import numpy as np

In [7]:
import os
import zipfile

with zipfile.ZipFile("cleaned_data.zip", "r") as zip_ref:
    zip_ref.extractall("cleaned")

print("Cleaned datasets extracted successfully!")
print(os.listdir("cleaned"))

Cleaned datasets extracted successfully!
['sellers.csv', 'order_items.csv', 'payments.csv', 'customers.csv', 'geolocation.csv', 'products.csv', 'orders.csv', 'reviews.csv', 'category_translation.csv']


In [ ]:
customers = pd.read_csv("../data/cleaned/customers.csv")
order_items = pd.read_csv("../data/cleaned/order_items.csv")
orders = pd.read_csv("../data/cleaned/orders.csv")
payments = pd.read_csv("../data/cleaned/payments.csv", engine="python")
reviews = pd.read_csv("../data/cleaned/reviews.csv")
products = pd.read_csv("../data/cleaned/products.csv")
sellers = pd.read_csv("../data/cleaned/sellers.csv")
category_translation = pd.read_csv("../data/cleaned/category_translation.csv")

print("Datasets loaded successfully!")

Datasets loaded successfully!


In [9]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

print("Order date columns converted successfully!")

Order date columns converted successfully!


In [10]:
order_value = (
    order_items
    .groupby("order_id")
    .agg(
        product_value=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

order_value["total_order_value"] = (
    order_value["product_value"] +
    order_value["freight_value"]
)

order_value.head()

,order_id,product_value,freight_value,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04


In [11]:
orders = orders.merge(
    order_value[["order_id", "product_value", "freight_value", "total_order_value"]],
    on="order_id",
    how="left"
)

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,product_value,freight_value,total_order_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,29.99,8.72,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,118.70,22.76,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,159.90,19.22,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,45.00,27.20,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,19.90,8.72,28.62


In [13]:
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

orders["delivery_days"] = orders["delivery_days"].round(2)

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

orders["delivery_delay_days"] = orders["delivery_delay_days"].round(2)

orders[
    [
        "order_id",
        "delivery_days",
        "delivery_delay_days"
    ]
].head()

,order_id,delivery_days,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,8.44,-7.11
1,53cdb2fc8bc7dce0b6741e2150273451,13.78,-5.36
2,47770eb9100c2d0c44946d9cf07ec65d,9.39,-17.25
3,949d5b44dbf5de918fe9c16f97b45f8a,13.21,-12.98
4,ad21c59c0840e6cb83a9ceb5573f8159,2.87,-9.24


In [14]:
customers["customer_order_count"] = (
    orders.groupby("customer_id")["order_id"]
    .nunique()
    .reindex(customers["customer_id"])
    .fillna(0)
    .values
)

customer_spending = (
    orders.groupby("customer_id")["total_order_value"]
    .sum()
    .reindex(customers["customer_id"])
    .fillna(0)
)

customers["customer_total_spending"] = customer_spending.values

customers["customer_aov"] = np.where(
    customers["customer_order_count"] > 0,
    customers["customer_total_spending"] /
    customers["customer_order_count"],
    0
)

customers["is_repeat_customer"] = np.where(
    customers["customer_order_count"] > 1,
    1,
    0
)

customers[
    [
        "customer_id",
        "customer_order_count",
        "customer_total_spending",
        "customer_aov",
        "is_repeat_customer"
    ]
].head()

,customer_id,customer_order_count,customer_total_spending,customer_aov,is_repeat_customer
0,06b8999e2fba1a1fbc88172c00ba8bc7,1,146.87,146.87,0
1,18955e83d337fd6b2def6b18a428ac77,1,335.48,335.48,0
2,4e7b3e00288586ebd08712fdd0374a03,1,157.73,157.73,0
3,b2b6027bc5c5109e529d4dc6358b12c3,1,173.30,173.30,0
4,4f2d8ab171c80ec8364f7c12e35b23ad,1,252.25,252.25,0


In [15]:
seller_features = (
    order_items.groupby("seller_id")
    .agg(
        seller_order_count=("order_id", "nunique"),
        seller_revenue=("price", "sum")
    )
    .reset_index()
)

sellers = sellers.merge(
    seller_features,
    on="seller_id",
    how="left"
)

sellers["seller_order_count"] = sellers["seller_order_count"].fillna(0)
sellers["seller_revenue"] = sellers["seller_revenue"].fillna(0)

sellers[
    [
        "seller_id",
        "seller_order_count",
        "seller_revenue"
    ]
].head()

,seller_id,seller_order_count,seller_revenue
0,3442f8959a84dea7ee197c632cb2df15,3,218.70
1,d1b65fc7debc3361ea86b5f14c68d2e2,40,11703.07
2,ce3ad9de960102d0677a81f5d0bb7b2d,1,158.00
3,c0f3eea2e14555b6faeea3dd58c1b1c3,1,79.99
4,51a04a8a6bdcb23deccc82b0b80742cf,1,167.99


In [16]:
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("unknown")
)

products = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

products["product_category_name_english"] = (
    products["product_category_name_english"]
    .fillna(products["product_category_name"])
)

product_sales = (
    order_items.groupby("product_id")
    .agg(
        product_order_count=("order_id", "nunique"),
        product_units_sold=("order_item_id", "count"),
        product_revenue=("price", "sum")
    )
    .reset_index()
)

products = products.merge(
    product_sales,
    on="product_id",
    how="left"
)

products["product_order_count"] = products["product_order_count"].fillna(0)
products["product_units_sold"] = products["product_units_sold"].fillna(0)
products["product_revenue"] = products["product_revenue"].fillna(0)

products[
    [
        "product_id",
        "product_category_name_english",
        "product_order_count",
        "product_units_sold",
        "product_revenue"
    ]
].head()

,product_id,product_category_name_english,product_order_count,product_units_sold,product_revenue
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,1,1,10.91
1,3aa071139cb16b67ca9e5dea641aaa2f,art,1,1,248.00
2,96bd76ec8810374ed1b65e291975717f,sports_leisure,1,1,79.80
3,cef67bcfe19066a932b7673e239eb23d,baby,1,1,112.30
4,9dc1a7de274444849c219cff195d0b71,housewares,1,1,37.90


In [17]:
reviews["review_score"] = pd.to_numeric(
    reviews["review_score"],
    errors="coerce"
)

review_features = (
    reviews.groupby("order_id")
    .agg(
        review_score=("review_score", "mean")
    )
    .reset_index()
)

orders = orders.merge(
    review_features,
    on="order_id",
    how="left"
)

orders["review_score"] = orders["review_score"].fillna(
    orders["review_score"].median()
)

orders[
    [
        "order_id",
        "order_status",
        "delivery_days",
        "delivery_delay_days",
        "review_score"
    ]
].head()

,order_id,order_status,delivery_days,delivery_delay_days,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,8.44,-7.11,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,13.78,-5.36,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,9.39,-17.25,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,13.21,-12.98,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2.87,-9.24,5.0


In [18]:
orders["is_delayed"] = np.where(
    orders["delivery_delay_days"] > 0,
    1,
    0
)

orders["is_delivered"] = np.where(
    orders["order_status"] == "delivered",
    1,
    0
)

orders[
    [
        "order_id",
        "delivery_delay_days",
        "is_delayed",
        "is_delivered",
        "review_score"
    ]
].head()

,order_id,delivery_delay_days,is_delayed,is_delivered,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,-7.11,0,1,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,-5.36,0,1,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,-17.25,0,1,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,-12.98,0,1,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,-9.24,0,1,5.0


In [19]:
print(orders.shape)
print(customers.shape)
print(sellers.shape)
print(products.shape)

(99441, 16)
(99441, 9)
(3095, 6)
(32951, 13)


In [20]:
print(orders.isnull().sum())
print(customers.isnull().sum())
print(sellers.isnull().sum())
print(products.isnull().sum())

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
product_value                     775
freight_value                     775
total_order_value                 775
delivery_delay_days              2965
delivery_days                    2965
review_score                        0
is_delayed                          0
is_delivered                        0
dtype: int64
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
customer_order_count        0
customer_total_spending     0
customer_aov                0
is_repeat_customer          0
dtype: int64
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state

In [21]:
import os

os.makedirs("engineered", exist_ok=True)

orders.to_csv("engineered/orders_engineered.csv", index=False)
customers.to_csv("engineered/customers_engineered.csv", index=False)
sellers.to_csv("engineered/sellers_engineered.csv", index=False)
products.to_csv("engineered/products_engineered.csv", index=False)
order_items.to_csv("engineered/order_items_engineered.csv", index=False)
reviews.to_csv("engineered/reviews_engineered.csv", index=False)

print("Feature-engineered datasets saved successfully!")

Feature-engineered datasets saved successfully!


In [22]:
print(os.listdir("engineered"))

['order_items_engineered.csv', 'orders_engineered.csv', 'products_engineered.csv', 'reviews_engineered.csv', 'customers_engineered.csv', 'sellers_engineered.csv']


In [23]:
import shutil

shutil.make_archive(
    "engineered_data",
    "zip",
    "engineered"
)

print("engineered_data.zip created successfully!")

engineered_data.zip created successfully!
